In [ ]:
######## Step 1&2&3: Input and extract the corresponding sequences from genome files and generate multi-FASTA file ###########

import pandas as pd
import subprocess
import os

CSV_FILE = "./Orthologs/orthlogNames/shared_candidates_intersection.csv"
OUTPUT_DIR = "./MSA/msa_input_fastas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

geno_mapping = {
    "genotype1": {"path": "./genotype1.cds.fa", "name": "genotype1"},
    "genotype2": {"path": "./genotype2.cds.fa", "name": "genotype2"},
    "genotype3": {"path": "./genotype3.cds.fa", "name": "genotype3"}
}

def extract_and_merge():
    print("Data loading..")
    df = pd.read_csv(CSV_FILE)
    
    # Extract the existing genotype column in df matching the new mapping
    genotypes_in_df = [col for col in df.columns if col in geno_mapping.keys()]
    
    for index, row in df.iterrows():
        qseqid = str(row['qseqid']).strip()
        safe_qseqid = qseqid.replace('|', '_').replace(':', '_')
        
        output_fasta = os.path.join(OUTPUT_DIR, f"{safe_qseqid}_combined.fasta")
        extracted_count = 0
        
        with open(output_fasta, 'w') as out_f:
            for geno in genotypes_in_df:
                cell_value = str(row[geno])
                
                # Run if the data exists
                if cell_value.lower() != 'nan' and cell_value.lower() != 'absent':
                    # divide the subgenome sequences sublisted with ,
                    seq_ids = [s.strip() for s in cell_value.split(',')]
                    fasta_path = geno_mapping[geno]["path"]
                    display_name = geno_mapping[geno]["name"]
                    
                    for seq_id in seq_ids:
                        if not seq_id: continue
                        
                        # Extract corresponding sequences using samtools faidx
                        cmd = ["samtools", "faidx", fasta_path, seq_id]
                        try:
                            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
                            fasta_text = result.stdout.strip()
                            
                            if fasta_text:
                                lines = fasta_text.split('\n')
                                
                                lines[0] = f">{display_name}_{seq_id}" 
                    
                                out_f.write('\n'.join(lines) + '\n')
                                extracted_count += 1
                            else:
                                print(f" Cannot find {seq_id} in {geno} ({display_name}).")
                                
                        except subprocess.CalledProcessError:
                            print(f"Samtools running failed: {seq_id} cannot be extracted.")
        
        print(f"{safe_qseqid} are extracted: (Total: {extracted_count}) -> {output_fasta}")

if __name__ == "__main__":
    try:
        subprocess.run(["samtools", "--version"], capture_output=True, check=True)
        extract_and_merge()
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("No samtools")

In [ ]:
########### Step 4. Alignment using MAFFT ##############

import os
import glob
import subprocess

INPUT_DIR = "./MSA/msa_input_fastas"
OUTPUT_DIR = "./MSA/msa_aligned_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run_mafft_alignment():
    fasta_files = glob.glob(os.path.join(INPUT_DIR, "*_combined.fasta"))
    
    if not fasta_files:
        print("No file for alignment are found.")
        return

    print(f"Total: {len(fasta_files)}are being aligned..")
    
    success_count = 0
    
    for i, input_fasta in enumerate(fasta_files, 1):
        base_name = os.path.basename(input_fasta).replace("_combined.fasta", "")
        output_aln = os.path.join(OUTPUT_DIR, f"{base_name}_aligned.fasta")
        
        cmd = ["mafft", "--auto", "--thread", "-1", input_fasta]
        
        try:
            with open(output_aln, "w") as out_f:
                subprocess.run(cmd, stdout=out_f, stderr=subprocess.DEVNULL, check=True)
            
            print(f"Processed: [{i}/{len(fasta_files)}] {base_name} completed", end="", flush=True)
            success_count += 1
            
        except subprocess.CalledProcessError:
            print(f"Error in: {base_name}")
            
    print(f"MAFFT alignment completed: (Total: {success_count}/{len(fasta_files)})")
    print(f"Saved in: {OUTPUT_DIR}")

if __name__ == "__main__":
    try:
        subprocess.run(["mafft", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        run_mafft_alignment()
    except FileNotFoundError:
        print("No MAFFT.")

In [ ]:
############### Step 5. Parsing ################

import os
import glob
import pandas as pd
from Bio import AlignIO


INPUT_DIR = "./MSA/msa_aligned_results"
OUTPUT_DIR = "./MSA/variant_summaries"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def parse_msa_to_table():
    aln_files = glob.glob(os.path.join(INPUT_DIR, "*_aligned.fasta"))
    
    if not aln_files:
        print("Cannot find aligned file.")
        return

    print(f"From total {len(aln_files)}of aligned file, trying extracting variations (SNP/InDel)..")
    
    for aln_file in aln_files:
        base_name = os.path.basename(aln_file).replace("_aligned.fasta", "")
        
        # Read the aligned file through biopython
        try:
            alignment = AlignIO.read(aln_file, "fasta")
        except Exception as e:
            print(f"{base_name} cannot read: {e}")
            continue
            
        variants = []
        # extract the seq ID in aligned file
        seq_ids = [record.id for record in alignment]
        aln_length = alignment.get_alignment_length()
        
        for i in range(aln_length):
            column = alignment[:, i].upper()
            unique_chars = set(column)
            
            # Define Variation if >1 unique character has been detected
            if len(unique_chars) > 1:
                # - : InDel, else : SNP (substitution)
                var_type = "InDel" if "-" in unique_chars else "SNP"
                
                row_data = {
                    "Locus": base_name,
                    "Alignment_Position": i + 1,  # 1-based index
                    "Variant_Type": var_type
                }
                
                for j, seq_id in enumerate(seq_ids):
                    row_data[seq_id] = column[j]
                    
                variants.append(row_data)
                
       # Save the file          
        if variants:
            df = pd.DataFrame(variants)
            out_csv = os.path.join(OUTPUT_DIR, f"{base_name}_variants.csv")
            df.to_csv(out_csv, index=False)
            print(f"{base_name}: Total {len(variants)} variations detected")
        else:
            print(f" {base_name}: No variation.")

    print(f"Parsing completed, saved in: {OUTPUT_DIR}")

if __name__ == "__main__":
    parse_msa_to_table()